# NeuroScan 3D Tumor Segmentation - Prediction Visualization
This notebook automatically discovers all predicted segmentation masks in , loads corresponding original MRI sequences from , computes volumetric measurements (WT, TC, ET), and renders multiplanar 2D orthogonal views and interactive slice sliders.

In [ ]:
import re
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

PREDICT_DIR = Path(".").resolve()
DATA_DIR = (PREDICT_DIR.parent / "data").resolve()
print(f"Predict Dir: {PREDICT_DIR}")
print(f"Data Dir:    {DATA_DIR}")

In [ ]:
# BraTS Label Colormaps:
# 0: Background (Transparent)
# 1: Enhancing Tumor (ET) -> Yellow (#FFD700)
# 2: Necrotic and Non-Enhancing Core (NCR) -> Red (#E53E3E)
# 3: Peritumoral Edema (ED) -> Blue (#3182CE)

cmap_dict = {
    0: (0.0, 0.0, 0.0, 0.0),      # Transparent
    1: (1.0, 0.843, 0.0, 0.85),   # ET: Yellow
    2: (0.898, 0.243, 0.243, 0.85),# NCR: Red
    3: (0.192, 0.509, 0.808, 0.75) # ED: Blue
}
custom_cmap = mcolors.ListedColormap([cmap_dict[i] for i in range(4)])
norm_seg = mcolors.BoundaryNorm(boundaries=[-0.5, 0.5, 1.5, 2.5, 3.5], ncolors=4)

LEGEND_PATCHES = [
    mpatches.Patch(color="#FFD700", label="Enhancing Tumor (ET)"),
    mpatches.Patch(color="#E53E3E", label="Necrotic Core (NCR)"),
    mpatches.Patch(color="#3182CE", label="Peritumoral Edema (ED)"),
]

In [ ]:
def discover_predictions_and_scans():
    seg_files = sorted(list(PREDICT_DIR.glob("*_seg.nii.gz")) + list(PREDICT_DIR.glob("*_seg.nii")))
    cases = []
    
    for seg_p in seg_files:
        case_id = seg_p.name.replace("_seg.nii.gz", "").replace("_seg.nii", "")
        
        # Find corresponding raw scans in data_dir
        scans = {}
        for mod in ["flair", "t1", "t1ce", "t2"]:
            candidates = list(DATA_DIR.rglob(f"{case_id}_{mod}.nii*"))
            if candidates:
                scans[mod] = candidates[0]
                
        cases.append({
            "case_id": case_id,
            "seg_path": seg_p,
            "scans": scans
        })
    return cases

cases = discover_predictions_and_scans()
print(f"Discovered {len(cases)} cases with predictions in {PREDICT_DIR}: {[c['case_id'] for c in cases]}")

# Compute volume statistics
records = []
for c in cases:
    seg_nii = nib.load(str(c["seg_path"]))
    seg_arr = seg_nii.get_fdata().astype(np.uint8)
    
    # Voxel volume in cm3 (1 mm3 = 0.001 cm3)
    voxel_vol_cm3 = np.prod(seg_nii.header.get_zooms()[:3]) / 1000.0
    
    et_vox = int((seg_arr == 1).sum())
    ncr_vox = int((seg_arr == 2).sum())
    ed_vox = int((seg_arr == 3).sum())
    tc_vox = et_vox + ncr_vox
    wt_vox = et_vox + ncr_vox + ed_vox
    
    records.append({
        "Case ID": c["case_id"],
        "WT Volume (cm³)": round(wt_vox * voxel_vol_cm3, 2),
        "TC Volume (cm³)": round(tc_vox * voxel_vol_cm3, 2),
        "ET Volume (cm³)": round(et_vox * voxel_vol_cm3, 2),
        "ED Volume (cm³)": round(ed_vox * voxel_vol_cm3, 2),
        "WT Voxels": wt_vox,
        "Dimensions": str(seg_arr.shape)
    })

df_summary = pd.DataFrame(records)
display(df_summary)

In [ ]:
def plot_max_tumor_slices(case_info, base_modality="flair", alpha=0.6):
    case_id = case_info["case_id"]
    seg_nii = nib.load(str(case_info["seg_path"]))
    seg_arr = seg_nii.get_fdata().astype(np.uint8)
    
    scan_path = case_info["scans"].get(base_modality) or list(case_info["scans"].values())[0]
    scan_nii = nib.load(str(scan_path))
    scan_arr = scan_nii.get_fdata(dtype=np.float32)
    
    # Normalize base image for contrast
    scan_arr = np.clip(scan_arr / (np.percentile(scan_arr[scan_arr > 0], 99) if (scan_arr > 0).any() else 1.0), 0, 1)
    
    # Find slice with maximum tumor voxels on each axis
    tumor_mask = seg_arr > 0
    
    ax_axial_idx = int(np.argmax(tumor_mask.sum(axis=(0, 1)))) if tumor_mask.any() else seg_arr.shape[2] // 2
    ax_coronal_idx = int(np.argmax(tumor_mask.sum(axis=(0, 2)))) if tumor_mask.any() else seg_arr.shape[1] // 2
    ax_sagittal_idx = int(np.argmax(tumor_mask.sum(axis=(1, 2)))) if tumor_mask.any() else seg_arr.shape[0] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"Case: {case_id} (Base: {base_modality.upper()})", fontsize=15, fontweight="bold", y=1.02)
    
    # 1. Axial View (Z-slice)
    axes[0].imshow(np.rot90(scan_arr[:, :, ax_axial_idx]), cmap="gray")
    axes[0].imshow(np.rot90(seg_arr[:, :, ax_axial_idx]), cmap=custom_cmap, norm=norm_seg, alpha=alpha)
    axes[0].set_title(f"Axial (Z = {ax_axial_idx})")
    axes[0].axis("off")
    
    # 2. Coronal View (Y-slice)
    axes[1].imshow(np.rot90(scan_arr[:, ax_coronal_idx, :]), cmap="gray")
    axes[1].imshow(np.rot90(seg_arr[:, ax_coronal_idx, :]), cmap=custom_cmap, norm=norm_seg, alpha=alpha)
    axes[1].set_title(f"Coronal (Y = {ax_coronal_idx})")
    axes[1].axis("off")
    
    # 3. Sagittal View (X-slice)
    axes[2].imshow(np.rot90(scan_arr[ax_sagittal_idx, :, :]), cmap="gray")
    axes[2].imshow(np.rot90(seg_arr[ax_sagittal_idx, :, :]), cmap=custom_cmap, norm=norm_seg, alpha=alpha)
    axes[2].set_title(f"Sagittal (X = {ax_sagittal_idx})")
    axes[2].axis("off")
    
    fig.legend(handles=LEGEND_PATCHES, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.05), fontsize=12)
    plt.tight_layout()
    plt.show()

for c in cases:
    plot_max_tumor_slices(c, base_modality="flair")

In [ ]:
# Interactive 3D Slice Explorer with Slider
case_map = {c["case_id"]: c for c in cases}

def interactive_viewer(case_id, plane, modality, slice_idx, alpha):
    if not case_id or case_id not in case_map:
        return
    c = case_map[case_id]
    seg_arr = nib.load(str(c["seg_path"])).get_fdata().astype(np.uint8)
    scan_path = c["scans"].get(modality.lower()) or list(c["scans"].values())[0]
    scan_arr = nib.load(str(scan_path)).get_fdata(dtype=np.float32)
    scan_arr = np.clip(scan_arr / (np.percentile(scan_arr[scan_arr > 0], 99) if (scan_arr > 0).any() else 1.0), 0, 1)
    
    if plane == "Axial":
        max_s = seg_arr.shape[2] - 1
        s = min(slice_idx, max_s)
        img_slice = np.rot90(scan_arr[:, :, s])
        seg_slice = np.rot90(seg_arr[:, :, s])
    elif plane == "Coronal":
        max_s = seg_arr.shape[1] - 1
        s = min(slice_idx, max_s)
        img_slice = np.rot90(scan_arr[:, s, :])
        seg_slice = np.rot90(seg_arr[:, s, :])
    else: # Sagittal
        max_s = seg_arr.shape[0] - 1
        s = min(slice_idx, max_s)
        img_slice = np.rot90(scan_arr[s, :, :])
        seg_slice = np.rot90(seg_arr[s, :, :])
        
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img_slice, cmap="gray")
    ax.imshow(seg_slice, cmap=custom_cmap, norm=norm_seg, alpha=alpha)
    ax.set_title(f"{case_id} | {plane} Slice {s}/{max_s} ({modality})", fontsize=13, fontweight="bold")
    ax.axis("off")
    ax.legend(handles=LEGEND_PATCHES, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08))
    plt.show()

if cases:
    widgets.interact(
        interactive_viewer,
        case_id=widgets.Dropdown(options=list(case_map.keys()), value=cases[0]["case_id"], description="Case:"),
        plane=widgets.RadioButtons(options=["Axial", "Coronal", "Sagittal"], value="Axial", description="Plane:"),
        modality=widgets.Dropdown(options=["FLAIR", "T1", "T1ce", "T2"], value="FLAIR", description="Sequence:"),
        slice_idx=widgets.IntSlider(min=0, max=154, step=1, value=75, description="Slice:", continuous_update=False),
        alpha=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.6, description="Opacity:")
    )